# 🔑 Kunci Jawaban — Kuis Bab 11 (LLM API & Orkestrasi)

> **Buka file ini HANYA SETELAH** mencoba kuis sendiri. Membaca jawaban sebelum
> berjuang = nol pembelajaran (ilusi kompetensi).

Untuk tiap soal: jawaban → **kenapa** → kode referensi yang bisa dijalankan.
Fokusnya tetap *kenapa*.


## Setup (sama dengan kuis — untuk menjalankan kode referensi)


In [ ]:
PESAN = [{'role': 'user', 'content': 'Halo'}]


class GalatAPI(Exception):
    def __init__(self, jenis, pesan='gagal'):
        super().__init__(f'[{jenis}] {pesan}')
        self.jenis = jenis


class GalatRateLimit(GalatAPI):
    def __init__(self, pesan='gagal'):
        super().__init__('rate_limit', pesan)


class GalatServer(GalatAPI):
    def __init__(self, pesan='gagal'):
        super().__init__('server_error', pesan)


class GalatTimeout(GalatAPI):
    def __init__(self, pesan='gagal'):
        super().__init__('timeout', pesan)


class GalatPermintaanBuruk(GalatAPI):
    def __init__(self, pesan='gagal'):
        super().__init__('bad_request', pesan)


class GalatSemuaPercobaan(GalatAPI):
    def __init__(self, riwayat):
        super().__init__('semua_gagal', 'semua percobaan gagal')
        self.riwayat = list(riwayat)


JENIS_TRANSIEN = ('rate_limit', 'server_error', 'timeout')
GALAT = {'rate_limit': GalatRateLimit, 'server_error': GalatServer,
         'timeout': GalatTimeout, 'bad_request': GalatPermintaanBuruk}


class ProviderUji:
    """Provider palsu: status panggilan ke-n dari `rencana` (habis -> ok)."""

    def __init__(self, rencana=(), latency_ms=200):
        self.rencana = list(rencana)
        self.latency_ms = latency_ms
        self.panggilan = 0

    def panggil(self, messages=None):
        self.panggilan += 1
        status = self.rencana.pop(0) if self.rencana else 'ok'
        if status != 'ok':
            raise GALAT[status]('uji')
        return {'teks': 'ok', 'latency_ms': self.latency_ms, 'attempts': self.panggilan}


TOOLS = {
    'get_cuaca': {'argumen': {'kota': 'Jakarta', 'satuan': 'celsius'}},
    'hitung': {'argumen': {'ekspresi': '1 + 1'}},
}

print('setup kunci siap:', [c.__name__ for c in GALAT.values()])


## Bagian A — Pilihan Ganda


### Soal 1 — Jawaban: **c) langsung lempar; percobaan ulang tidak mengubah apa pun**

`bad_request`/model-tidak-dikenal adalah kegagalan **deterministik**: permintaan
yang sama akan ditolak dengan cara yang sama. Mengulanginya berarti membayar N
percobaan untuk satu kesalahan, menambah latensi pengguna, dan menaikkan beban
provider — tanpa peluang membaik sedikit pun.

Bukti dari lab Bagian 2.2: `panggil_dengan_retry(..., 'model_ajaib', ...)`
melempar `GalatPermintaanBuruk` dan `provider.jumlah_panggilan['model_ajaib']`
tetap **1**. Bandingkan dengan `besar` yang transien: 2 percobaan, +1000 ms.

Kenapa opsi lain salah:

- **a)** dan **b)** membakar biaya/latensi untuk kegagalan yang pasti berulang.
- **d)** "ganti model besar supaya lebih pintar" salah kategori: masalahnya
  permintaan tidak valid (nama model/skema), bukan kurang pintar.

Aturan praktis: **retry hanya untuk galat yang bisa hilang sendiri**
(`rate_limit`, `server_error`, `timeout`, error jaringan).


### Soal 2 — Jawaban: **b) supaya worst-case latency tetap dapat diterima**

Backoff eksponensial ada untuk memberi provider waktu pulih tanpa membanjiri.
Tapi tanpa batas atas, percobaannya sendiri menjadi masalah: 1000·2^(n-1) ms =
percobaan ke-10 sekitar 8,5 menit menunggu di depan pengguna.

Di lab Bagian 2.3: `[backoff_ms(n) for n in range(1, 6)]` =
`[1000, 2000, 4000, 8000, 8000]` — dua angka terakhir SAMA, karena `maks_ms=8000`.

Kenapa opsi lain salah:

- **a)** batas atas bukan soal kerapian kode, tapi anggaran latency.
- **c)** provider tidak peduli jeda klien; kamu yang menanggungnya.
- **d)** sebaliknya: batas atas tidak menambah jumlah percobaan.

Di produksi: batas atas + **jitter** acak supaya ribuan klien tidak bangun
bersamaan (thundering herd), lalu **circuit breaker** kalau provider sedang
down total — berhenti mencoba lebih murah daripada mencoba terus.


### Soal 3 — Jawaban: **a) ketersediaan naik, dibayar dengan konsistensi & kualitas**

Fallback adalah trade-off eksplisit, bukan perbaikan gratis. Di lab Bagian 3,
rantai `besar → mini` menyelamatkan permintaan (lompatan ke-1) dengan biaya
~26x lebih murah — tapi modelnya berbeda, jadi gaya output dan kualitas
penalarannya berbeda juga.

Kenapa itu penting operasionalnya: pada sistem yang hasilnya diparse/dibandingkan
(JSON ber-skema, skor), jawaban dari model berbeda bisa **gagal skema** dan
memicu retry tambahan — hasilnya tampak sukses, biayanya dua kali. Karena itu
`dicoba`/`lompatan` harus dilog dan dibaca, bukan disembunyikan.

Kenapa opsi lain salah:

- **b)** kualitas TIDAK sama antar model; itu inti trade-off-nya.
- **c)** kalau tak ada trade-off, fallback tak perlu dirancang hati-hati.
- **d)** justru sebaliknya: tugas kreatif paling toleran turun-kualitas; ekstraksi
  ber-skema ketat paling tidak toleran (lebih baik error 503).


### Soal 4 — Jawaban: **b) model + pesan + parameter generasi (dan versi prompt/konteks)**

Cache yang salah kunci = bug yang menyajikan jawaban lama dengan percaya diri.
Semua yang mengubah output harus masuk kunci: model, isi pesan, dan parameter
generasi (`temperature`, `top_p`, `max_tokens`, `response_format`).

Di lab Bagian 4.2: `kunci('mini', PESAN)` berbeda dari `kunci('sedang', PESAN)`,
dan berbeda lagi ketika `temperature` 0.0 vs 1.0.

Kenapa opsi lain salah:

- **a)** prompt sama + model/parameter beda = jawaban beda.
- **c)** id pengguna bukan penentu isi jawaban (kecuali kamu SENGAJA
  mempersonalisasi — dan itu justru alasan menambah tenant id).
- **d)** timestamp bukan bagian dari identitas permintaan; TTL-lah yang
  mengurus kesegaran.

Dua tambahan wajib di produksi: **versi prompt** (prompt berubah → jawaban boleh
berubah) dan **konteks retrieval** (Bab 12) — plus tenant id untuk data sensitif
(jangan pernah cache lintas tenant).


### Soal 5 — Jawaban: **d) satu permintaan bisa berputar tanpa henti**

Loop tool berhenti karena **tiga** hal: model tidak lagi minta tool, `maks_iterasi`
tercapai, atau galat tool dikirim balik sebagai pesan `tool`. Hapus `maks_iterasi`
dan satu-satunya jaring pengaman untuk skenario "model terus meminta tool" hilang.

Di lab Bagian 6.2: `loop_tool(..., 'Berapa suhu Jakarta hari ini?', maks_iterasi=1)`
berhenti dengan `berhenti='maks_iterasi'` dan `jawaban=None` — laporan yang jujur,
bukan menggantung. Loop normal dengan dua tool selesai di iterasi ke-3.

Kenapa opsi lain salah:

- **a)** perilaku model TIDAK dijamin; asumsi "modelnya pintar" bukan kontrol.
- **b)** de-dup tool sudah ditangani terpisah (dicek dari assistant `tool_calls`,
  bukan dari teks prompt) — bukan alasan batas iterasi.
- **c)** validitas hasil tool dijamin oleh validasi argumen, bukan oleh jumlah
  iterasi.

Praktik produksi: batas iterasi + batas token/anggaran per permintaan + timeout
total — tiga batas, karena masing-masing menangkap kegagalan berbeda.


### Soal 6 — Jawaban: **c) waktu sampai token pertama terlihat**

Streaming memindahkan **waktu tunggu**, tidak menghapusnya. Di lab Bagian 5:
TTFT 120 ms, total 340 ms — total prosesnya identik dengan non-streaming.
Yang berubah: pengguna melihat kata pertama ~65% lebih cepat.

Kenapa opsi lain salah:

- **a)** throughput/total waktu tidak berubah karena streaming.
- **b)** biaya token tidak berkurang; jumlah token yang dibayar sama.
- **d)** kualitas jawaban tidak naik karena dikirim per chunk.

Kapan streaming **tidak** layak: output JSON pendek yang baru berguna setelah
utuh, tugas batch/async yang tidak ditunggu manusia, atau ketika parsing
membutuhkan hasil lengkap. Di situ streaming hanya menambah kompleksitas.


## Bagian B — Coding


### Soal 7 — `backoff_ms`

Satu baris, tapi ada dua jebakan: pangkatnya `n - 1` (percobaan pertama = basis,
bukan basis·faktor), dan hasilnya harus **dibatasi** `maks_ms`.


In [ ]:
def backoff_ms(percobaan, basis_ms=1000, faktor=2, maks_ms=8000):
    return min(basis_ms * (faktor ** (percobaan - 1)), maks_ms)

print([backoff_ms(n) for n in range(1, 6)])
print([backoff_ms(n, 100, 3, 500) for n in range(1, 5)])   # batas bekerja


### Soal 8 — `panggil_dengan_retry`

Urutan keputusannya yang benar (dan mudah salah):

```
1. panggil provider
2. catat (percobaan, jenis) ke riwayat
3. galat FATAL?          -> raise sekarang
4. percobaan TERAKHIR?   -> raise GalatSemuaPercobaan (JANGAN menunggu dulu)
5. selain itu             -> sleep(backoff(percobaan)) lalu ulangi
```

Jebakan paling umum: menunggu **sebelum** memeriksa "ini percobaan terakhir?",
sehingga ada satu jeda sia-sia. Perhatikan juga `sleep_fn` — di test kita suntik
`tunggu.append`, jadi tidak ada detik nyata yang terbuang.


In [ ]:
def panggil_dengan_retry(provider, messages, maks_percobaan=4, basis_ms=1000,
                         faktor=2, maks_ms=8000, sleep_fn=None):
    riwayat, tunggu = [], []
    for percobaan in range(1, maks_percobaan + 1):
        try:
            hasil = provider.panggil(messages)
        except GalatAPI as e:
            riwayat.append((percobaan, e.jenis))
            if e.jenis not in JENIS_TRANSIEN:
                raise                      # fatal: tidak pernah diulang
            if percobaan == maks_percobaan:
                raise GalatSemuaPercobaan(riwayat) from e   # jangan menunggu lagi
            jeda = backoff_ms(percobaan, basis_ms, faktor, maks_ms)
            tunggu.append(jeda)
            if sleep_fn is not None:
                sleep_fn(jeda)
            continue
        hasil = dict(hasil)
        hasil['attempts'] = percobaan
        hasil['timpenungguan_ms'] = list(tunggu)
        hasil['total_ms'] = hasil['latency_ms'] + sum(tunggu)
        hasil['riwayat'] = list(riwayat)
        return hasil
    raise GalatSemuaPercobaan(riwayat)


tunggu = []
p1 = ProviderUji(['rate_limit', 'ok'])
print(panggil_dengan_retry(p1, PESAN, sleep_fn=tunggu.append))
print('jeda yang benar-benar dipakai:', tunggu)

for nama, rencana, kw in [('transien habis', ['rate_limit', 'server_error'],
                          {'maks_percobaan': 2}),
                          ('fatal', ['bad_request'], {}),
                          ('mentok di percobaan 1', ['rate_limit'],
                           {'maks_percobaan': 1})]:
    p = ProviderUji(rencana)
    try:
        panggil_dengan_retry(p, PESAN, sleep_fn=lambda ms: None, **kw)
        print(nama, '-> TIDAK melempar (salah?)')
    except GalatAPI as e:
        print(f'{nama:22s} -> {e.jenis} | panggilan ke provider: {p.panggilan}')


### Soal 9 — `validasi_argumen`

Prinsipnya: skema dibangun dari **contoh argumen** di `TOOLS`, jadi semua kunci
contoh dianggap wajib dan tipe nilainya jadi tipe yang diharapkan.
Urutan error dibuat stabil supaya bisa di-assert dan dipakai sebagai pesan retry
ke model: `missing:` dulu (urut kunci wajib), lalu `tak_dikenal:`/`type:`.

Jebakan klasik Python tetap berlaku kalau nanti kamu tambah tool bertipe integer:
`bool` adalah subclass `int`, jadi harus dikecualikan eksplisit.


In [ ]:
TIPE_JSON = {str: 'string', int: 'integer', float: 'number', bool: 'boolean'}


def _tipe_ok(nilai, tipe):
    if tipe == 'string':
        return isinstance(nilai, str)
    if tipe == 'integer':
        return isinstance(nilai, int) and not isinstance(nilai, bool)
    if tipe == 'number':
        return isinstance(nilai, (int, float)) and not isinstance(nilai, bool)
    if tipe == 'boolean':
        return isinstance(nilai, bool)
    return False


def validasi_argumen(nama, argumen, tools=TOOLS):
    if nama not in tools:
        return [f'tak_dikenal:{nama}']
    if not isinstance(argumen, dict):
        return ['type:<argumen>']
    contoh = tools[nama]['argumen']
    props = {k: TIPE_JSON.get(type(v), 'string') for k, v in contoh.items()}
    errors = []
    for k in contoh:                       # urutan wajib = urutan contoh
        if k not in argumen:
            errors.append(f'missing:{k}')
    for k, v in argumen.items():
        if k not in props:
            errors.append(f'tak_dikenal:{k}')
        elif not _tipe_ok(v, props[k]):
            errors.append(f'type:{k}')
    return errors


print(validasi_argumen('get_cuaca', {'kota': 'Bandung', 'satuan': 'celsius'}))
print(validasi_argumen('get_cuaca', {}))
print(validasi_argumen('get_cuaca', {'kota': 5, 'satuan': 'celsius'}))
print(validasi_argumen('get_cuaca', {'kota': 'a', 'satuan': 'c', 'x': 1}))
print(validasi_argumen('nuklir', {}))
print(validasi_argumen('get_cuaca', ['kota']))


### Soal 10 — `konsumsi_stream`

Tiga definisi yang harus persis: `jumlah_chunk` menghitung delta **non-kosong**
saja, `ttft_ms` diambil dari chunk non-kosong **pertama**, dan `total_ms` dari
chunk **terakhir** (termasuk chunk kosong penanda akhir). Iterasi terakhir ini
yang membuat `total_ms` = 340 ms di lab, bukan 320 ms.


In [ ]:
def konsumsi_stream(chunks):
    potongan, jumlah, ttft, terakhir = [], 0, None, 0
    for c in chunks:
        terakhir = c['t_ms']
        if c['delta']:
            jumlah += 1
            if ttft is None:
                ttft = c['t_ms']
            potongan.append(c['delta'])
    return {'teks': ''.join(potongan), 'jumlah_chunk': jumlah,
            'ttft_ms': ttft, 'total_ms': terakhir}


chunks = [{'delta': '', 't_ms': 10}, {'delta': 'Halo', 't_ms': 55},
          {'delta': ' dunia', 't_ms': 90}, {'delta': '', 't_ms': 110}]
print(konsumsi_stream(chunks))
print(konsumsi_stream([{'delta': '', 't_ms': 42}]))   # ttft None: belum ada kata
print(konsumsi_stream([]))                            # total_ms 0


---

Skor penuh? Lanjut ke `project-llmkit/` — di sana semua konsep ini disatukan
menjadi paket `llmkit` end-to-end dengan **107 test**: cost, retry, fallback,
cache, stream, tools, dan orkestrasi pipeline.
